In [43]:
import numpy as np
import torch
from scipy.linalg import svdvals

def calculate_similarity_r2(vectors):
   """
   Calculates the R² similarity measure from three input vectors.
   Parameters:
   vectors (list of np.ndarray): List of three vectors (1D arrays) of equal length.
   Returns:
   float: R² value indicating similarity (between 0 and 1).
   """
   matrix = np.column_stack(vectors)
   U, singular_values, VT = np.linalg.svd(matrix)
   sigma_1 = singular_values[0]
   print(f"Singular values: {singular_values}")
   r_squared = (sigma_1 ** 2) / np.sum(singular_values ** 2)
   return r_squared
# Example
v1 = np.array([-1, 0, 0])
v2 = np.array([1, 0, 0])
# v3 = np.array([0, 0, 0])       
r2_value = calculate_similarity_r2([v1, v2])
print(f"R² similarity: {r2_value:.4f}")

print(v1 @ v2) # v1 @ v2

Singular values: [1.41421356 0.        ]
R² similarity: 1.0000
-1


In [36]:
import torch
import time

D = 20000
v1 = torch.randn(D)
v2 = torch.randn(D)
v3 = torch.randn(D)
vectors = torch.stack([v1, v2, v3], dim=0)  # [3,D]

# ------------------------------
# PyTorch SVD
start = time.time()
U, S, V = torch.linalg.svd(vectors)
r2_1 = (S[0]**2) / torch.sum(S**2)
end = time.time()
print(f"PyTorch SVD R²: {r2_1.item():.6f}, time: {end-start:.4f}s")

# ------------------------------
# PyTorch svdvals
start = time.time()
S_only = torch.linalg.svdvals(vectors)
r2_2 = (S_only[0]**2) / torch.sum(S_only**2)
end = time.time()
print(f"PyTorch svdvals R²: {r2_2.item():.6f}, time: {end-start:.4f}s")

# ------------------------------
# Gram matrix
start = time.time()
G = vectors @ vectors.T  # [3,3]
eigvals = torch.linalg.eigvalsh(G)
r2_3 = (eigvals[-1]**2) / torch.sum(eigvals**2)
end = time.time()
print(f"Gram matrix approx R²: {r2_3.item():.6f}, time: {end-start:.4f}s")


PyTorch SVD R²: 0.338494, time: 1.6000s
PyTorch svdvals R²: 0.338494, time: 0.0081s
Gram matrix approx R²: 0.343688, time: 0.0239s


In [ ]:
def batch_r2_similarity_lidar_j(lidar, image, text):
    """
    Args:
        lidar: [B,D] tensor, LiDAR features
        image: [B,D] tensor, Image features
        text:  [B,D] tensor, Text features
    Returns:
        sim_matrix: [B,B] each element R² similarity
            sim_matrix[i,j] = R²([image[i], text[i], lidar[j]])
    """
    B, D = lidar.shape

    image_exp = image[:, None, :].expand(B, B, D)  # [B,1,D] -> [B,B,D]
    text_exp  = text[:, None, :].expand(B, B, D)   # [B,1,D] -> [B,B,D]
    lidar_exp = lidar[None, :, :].expand(B, B, D) # [1,B,D] -> [B,B,D]

    # stack 
    vectors = torch.stack([image_exp, text_exp, lidar_exp], dim=-2)

    # Gram 
    G = torch.matmul(vectors, vectors.transpose(-1,-2))

    # R2
    eigvals = torch.linalg.eigvalsh(G)  # [B,B,3]
    r2 = eigvals[..., -1] / eigvals.sum(dim=-1)  # [B,B]

    return r2

In [ ]:
def simple_pairwise_similarity(lidar, image, text):
    """
    Args:
        lidar, image, text: [B,D] L2-normalized tensors
    Returns:
        sim_matrix: [B,B], each element = average of three pairwise dot products
    """
    B, D = lidar.shape

    # expand [B,B,D]
    image_exp = image[:, None, :].expand(B, B, D)
    text_exp  = text[:, None, :].expand(B, B, D)
    lidar_exp = lidar[None, :, :].expand(B, B, D)

    # dot product
    dot_img_text  = torch.sum(image_exp * text_exp, dim=-1)
    dot_img_lidar = torch.sum(image_exp * lidar_exp, dim=-1)
    dot_text_lidar = torch.sum(text_exp * lidar_exp, dim=-1)

    # mean
    sim_matrix = (dot_img_text + dot_img_lidar + dot_text_lidar) / 3.0
    return sim_matrix

In [ ]:
B, D = 4, 8
lidar_features = torch.randn(B, D)
image_features = torch.randn(B, D)
text_features  = torch.randn(B, D)

# normalize
lidar_features = lidar_features / lidar_features.norm(dim=-1, keepdim=True)
image_features = image_features / image_features.norm(dim=-1, keepdim=True)
text_features  = text_features  / text_features.norm(dim=-1, keepdim=True)

sim_matrix = batch_r2_similarity_lidar_j(lidar_features, image_features, text_features)
print(sim_matrix.shape)  # [B,B]
print(sim_matrix)

sim_matrix_simple = simple_pairwise_similarity(lidar_features, image_features, text_features)
print(sim_matrix_simple.shape)  # [B,B]
print(sim_matrix_simple)

torch.Size([4, 4])
tensor([[0.4270, 0.4794, 0.4839, 0.4761],
        [0.5402, 0.5978, 0.5339, 0.6109],
        [0.4412, 0.5232, 0.5593, 0.4761],
        [0.6498, 0.5917, 0.4895, 0.5488]])
torch.Size([4, 4])
tensor([[ 0.0183,  0.1491, -0.1085, -0.2066],
        [-0.2060,  0.1222, -0.0961, -0.2455],
        [-0.0855,  0.0516, -0.1436, -0.1991],
        [ 0.4188,  0.3766,  0.1761, -0.1816]])
